# 🚀 UNIVERSAL AMBIK FAST TRAINER WITH TQDM (GOOGLE COLAB)
### 🏛️ Tích Hợp Thanh Tiến Trình TQDM Trực Quan & Tùy Chỉnh Số Câu Train Siêu Tốc
---
Notebook này đã được cập nhật tương thích 100% với phiên bản HuggingFace Transformers mới nhất (`processing_class` / `eval_strategy`).

## 🛠️ BƯỚC 1: CÀI ĐẶT THƯ VIỆN & KIỂM TRA GPU
* **Tác dụng:** Cài đặt các gói Deep Learning cần thiết kèm thư viện `tqdm` hiển thị thanh tiến trình.

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas numpy tqdm
!nvidia-smi

## 🎛️ BƯỚC 2: BẢNG ĐIỀU KHIỂN CẤU HÌNH (TÙY CHỈNH TẠI ĐÂY)
* **Tác dụng:** Tùy chỉnh số câu train (`MAX_SAMPLES = 100` hoặc `200` để train trong 15-30 giây) và đổi tên bất kỳ mô hình nào.

In [ ]:
# ==========================================================================
# ⚙️ BẢNG ĐIỀU KHIỂN CẤU HÌNH HUẤN LUYỆN (CHỈNH SỬA TẠI ĐÂY)
# ==========================================================================

# 1. TÊN MÔ HÌNH BẠN MUỐN DÙNG:
MODEL_NAME = "microsoft/deberta-v3-base"   # Khuyên dùng: Train 2 phút, chính xác cao nhất
# MODEL_NAME = "roberta-base"              # Lựa chọn 2: RoBERTa
# MODEL_NAME = "distilbert-base-uncased"   # Lựa chọn 3: DistilBERT siêu nhẹ
# MODEL_NAME = "Qwen/Qwen2.5-1.5B"          # Lựa chọn 4: Qwen 2.5 (Alibaba)

# 2. SỐ CÂU DỮ LIỆU MUỐN TRAIN:
# 👉 Đặt số nhỏ (vd: 100 hoặc 200 câu) để train SIÊU NHANH trong 15 - 30 giây!
# 👉 Đặt None nếu muốn train toàn bộ 2.000 câu.
MAX_SAMPLES = 200

# 3. SỐ EPOCHS (Chu kỳ huấn luyện): 2 hoặc 3 epochs là đủ
NUM_EPOCHS = 2

# 4. KÍCH THƯỚC BATCH:
BATCH_SIZE = 16

print(f"🎯 Mô hình: {MODEL_NAME}")
print(f"⚡ Giới hạn số câu train: {MAX_SAMPLES} mẫu | Epochs: {NUM_EPOCHS} | Batch size: {BATCH_SIZE}")

## 📂 BƯỚC 3: NẠP VÀ TRÍCH XUẤT DỮ LIỆU KÈM THANH TIẾN TRÌNH TQDM
* **Tác dụng:** Đọc `AmbiK_data.csv` với thanh tiến trình `tqdm` trực quan và đảm bảo mỗi nhãn có đủ số mẫu.

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

DATA_PATH = "AmbiK_data.csv"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("⚠️ Chưa tìm thấy file AmbiK_data.csv! Vui lòng tải file lên tab Files (bên trái Colab).")

df = pd.read_csv(DATA_PATH)
print(f"✅ Nạp thành công {len(df)} dòng dữ liệu gốc từ AmbiK.\n")

LABEL_MAP = {
    "preferences": 0,
    "common_sense_knowledge": 1,
    "safety": 2,
    "unambiguous": 3
}
ID2LABEL = {0: "Preferences", 1: "Common Sense", 2: "Safety", 3: "Unambiguous"}

records = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="⏳ Đang xử lý dữ liệu AmbiK"):
    amb_type = str(row.get('ambiguity_type', 'preferences')).strip().lower()
    env_full = str(row.get('environment_full', ''))
    amb_task = str(row.get('ambiguous_task', ''))
    unamb_task = str(row.get('unambiguous_direct', ''))
    
    label_id = LABEL_MAP.get(amb_type, 0)
    
    # Thêm câu lệnh mơ hồ
    if amb_task:
        records.append({"text": f"Instruction: {amb_task} | Scene: {env_full}", "label": label_id})
    # Thêm câu lệnh rõ ràng (Fast Route label = 3)
    if unamb_task:
        records.append({"text": f"Instruction: {unamb_task} | Scene: {env_full}", "label": 3})

dataset_df = pd.DataFrame(records)

# ⚡ Cắt giảm số mẫu an toàn (tối thiểu 15 mẫu mỗi nhãn để không bao giờ lỗi chia split)
if MAX_SAMPLES is not None and MAX_SAMPLES < len(dataset_df):
    samples_per_class = max(15, MAX_SAMPLES // 4)
    dataset_df = dataset_df.groupby('label', group_keys=False).apply(
        lambda x: x.sample(min(len(x), samples_per_class), random_state=42)
    ).reset_index(drop=True)
    print(f"\n⚡ ĐÃ RÚT GỌN TẬP DỮ LIỆU XUỐNG: {len(dataset_df)} MẪU ĐỂ TRAIN SIÊU NHANH!")
else:
    print(f"\n📊 Sử dụng toàn bộ: {len(dataset_df)} mẫu.")

print("Phân bố các nhãn:", dataset_df['label'].map(ID2LABEL).value_counts().to_dict())

## ✂️ BƯỚC 4: PHÂN CHIA TRAIN/VAL/TEST AN TOÀN & TOKENIZATION
* **Tác dụng:** Tự động kiểm tra số lượng mẫu và phân chia Train (70%), Validation (15%), Test (15%) an toàn tuyệt đối.

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 🛡️ PHÂN CHIA DỮ LIỆU AN TOÀN
min_class_count = dataset_df['label'].value_counts().min()
use_stratify = min_class_count >= 4

if use_stratify:
    train_df, temp_df = train_test_split(dataset_df, test_size=0.30, random_state=42, stratify=dataset_df['label'])
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])
else:
    train_df, temp_df = train_test_split(dataset_df, test_size=0.30, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
 
print(f"🔹 Tập Train: {len(train_df)} câu | Tập Val: {len(val_df)} câu | Tập Test: {len(test_df)} câu")

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

def tokenize_func(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = raw_datasets.map(tokenize_func, batched=True, desc="🔄 Tokenizing Datasets")
print("✅ Tokenization hoàn tất an toàn!")

## 🧠 BƯỚC 5: KHỞI TẠO MÔ HÌNH VÀ HÀM ĐÁNH GIÁ ACCURACY / F1
* **Tác dụng:** Khởi tạo kiến trúc phân loại 4 nhãn.

In [ ]:
import evaluate
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id={v: k for k, v in ID2LABEL.items()},
    trust_remote_code=True
)

if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.pad_token_id

metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = metric_acc.compute(predictions=preds, references=labels)["accuracy"]
    f1_macro = metric_f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {
        "accuracy": acc,
        "f1_macro": f1_macro
    }

## 🏋️ BƯỚC 6: TIẾN HÀNH HUẤN LUYỆN SIÊU TỐC (TƯƠNG THÍCH MỌI PHIÊN BẢN HUGGINGFACE)
* **Tác dụng:** Huấn luyện mô hình với cơ chế tự động tương thích phiên bản HuggingFace mới nhất.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

# Thiết lập TrainingArguments tương thích cả bản transformers mới và cũ
training_args_kwargs = {
    "output_dir": "./model_checkpoints",
    "save_strategy": "epoch",
    "learning_rate": 3e-5,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": 0.01,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1_macro",
    "fp16": True,
    "logging_steps": 5,
    "disable_tqdm": False,
    "report_to": "none"
}

try:
    training_args = TrainingArguments(eval_strategy="epoch", **training_args_kwargs)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy="epoch", **training_args_kwargs)

# Data Collator chuẩn hóa padding theo batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Khởi tạo Trainer với cơ chế tự động tương thích processing_class / tokenizer
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics
}

try:
    trainer = Trainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    try:
        trainer = Trainer(tokenizer=tokenizer, **trainer_kwargs)
    except TypeError:
        trainer = Trainer(**trainer_kwargs)

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN {len(train_df)} CÂU TRÊN GPU...\n")
trainer.train()

## 📈 BƯỚC 7: ĐÁNH GIÁ KẾT QUẢ TRÊN TẬP TEST ĐỘC LẬP
* **Tác dụng:** Đo lường độ chính xác Accuracy và Macro F1 sau khi train.

In [ ]:
test_results = trainer.evaluate(tokenized_datasets["test"])
print("=" * 60)
print(f"📊 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ({len(test_df)} CÂU TEST):")
print(f"🎯 Accuracy: {test_results['eval_accuracy']*100:.2f}%")
print(f"🎯 Macro F1: {test_results['eval_f1_macro']*100:.2f}%")
print("=" * 60)

## 🛡️ BƯỚC 8: HIỆU CHUẨN CONFORMAL PREDICTION (KNOWNO — NEURIPS 2023)
* **Tác dụng:** Dùng vòng lặp `tqdm` để tính toán phân vị $\hat{q}$ bảo đảm an toàn toán học $\ge 95\%$.

In [ ]:
import math
import torch
from tqdm.auto import tqdm

val_preds = trainer.predict(tokenized_datasets["validation"])
probs = torch.softmax(torch.tensor(val_preds.predictions), dim=1).numpy()
true_labels = val_preds.label_ids

scores = []
for p, y in tqdm(zip(probs, true_labels), total=len(true_labels), desc="🛡️ Đang tính Conformal Scores"):
    scores.append(1.0 - p[y])

alpha = 0.05 # Mức tin cậy 95%
n = len(scores)
quantile_level = min(math.ceil((n + 1) * (1 - alpha)) / n, 1.0)
q_hat = float(np.quantile(scores, quantile_level, method="higher"))

print(f"\n🛡️ Ngưỡng Phân vị Bảo giác Conformal Quantile: q_hat = {q_hat:.4f}")
print(f"✅ Đã hiệu chuẩn thành công cho tập dữ liệu với cam kết an toàn 95%!")

## 💾 BƯỚC 9: ĐÓNG GÓI & TẢI MÔ HÌNH VỀ MÁY TÍNH
* **Tác dụng:** Nén model thành file zip để tải về máy.

In [ ]:
SAVE_DIR = "./ambik_custom_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

!zip -r ambik_custom_model.zip ./ambik_custom_model

print("=" * 60)
print("🎉 HOÀN TẤT HUẤN LUYỆN VÀ ĐÓNG GÓI!")
print("Tải file 'ambik_custom_model.zip' từ thanh quản lý tệp (Files) bên trái của Colab.")
print("=" * 60)

from google.colab import files
files.download('ambik_custom_model.zip')